# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, not a dict

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print("Published: ", metadata.datePublished)

print("\nHigh-level Metadata Fields (pprint):")
fields = [f for f in dir(metadata) if not f.startswith('_')]
pprint.pprint(fields)

## 2. Data Overview
Review available record sets, fields, and their IDs.

<details>
<summary>
Print concise summary of available `recordSet`/record sets, and for each, list its fields.
</summary>
</details>

In [ ]:
# List Record Sets, their IDs, and Fields

print("Record Sets available in the dataset:")
all_record_sets = dataset.record_sets
if not all_record_sets:
    print("No record sets found in metadata. Attempt to extract from columns/fields...")
    # As per Croissant, some datasets may structure fields as top-level columns/files
columns = getattr(metadata, 'column', None)
if columns is not None:
    print("Top-level Columns detected in dataset (single table structure):")
    for col in columns:
        print(f"  - {col['@id']}: {col.get('name', col['@id'])}")
else:
    for rs in all_record_sets:
        print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', rs['@id'])}")
        if 'field' in rs:
            for field in rs['field']:
                print(f"    - Field @id: {field['@id']}, name: {field.get('name', field['@id'])}")

# For this FAIR² dataset, let's also dump the first available distribution
if hasattr(metadata, 'distribution'):
    print("\nDistributions (files/tables) in the dataset:")
    for dist in metadata.distribution:
        print(f"- Distribution @id: {dist['@id']}")
        if 'name' in dist:
            print(f"   name: {dist['name']}")
        if 'encodingFormat' in dist:
            print(f"   type: {dist['encodingFormat']}")


## 3. Data Extraction
Load data from the main distribution into a DataFrame for analysis. The primary data table is usually referenced by its `@id`; as seen above, we'll use the first CSV distribution as the main record set for demonstration.

Below, we will load all available record sets or, if none are defined, load data using the distribution/table `@id`.

In [ ]:
# List of record set or table @id(s)

# FAIR² appears to structure its main data as one table/file:
data_table_id = "https://api.app.sen.science/frontiers/7160411/794de323-23ec-4496-8bd5-d9c5b848dafe"

record_sets = [data_table_id]
dataframes = {}

for record_set in record_sets:
    print(f"Loading records for record set/table @id: {record_set}")
    records = list(dataset.records(record_set=record_set))
    if records:
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded DataFrame with shape {dataframes[record_set].shape}")
    else:
        print(f"No records found for this record set.")

if dataframes[record_sets[0]].shape[0] > 0:
    print("\nColumns in main data frame:")
    print(dataframes[record_sets[0]].columns.tolist())
    display(dataframes[record_sets[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming numeric distributions, and grouping records by demographic attributes.

Let's select a numeric field (e.g., GAD-7, PHQ-9, PSQ scores) for demonstration.

Referencing by column `@id` as found in the schema. If not available, field/column names are used as in the loaded DataFrame.

In [ ]:
# Choose a numeric field for demonstration, typically 'gad7_score' or a similar field for this mental health dataset
df = dataframes[data_table_id]
possible_numeric_cols = [c for c in df.columns if any(s in c.lower() for s in ['gad7', 'phq9', 'psq', 'score', 'age'])]
print("Numeric-related columns detected:", possible_numeric_cols)

# We'll use 'gad7_score' by default if it exists
numeric_field = None
for col in ['gad7_score', 'phq9_score', 'psq_score', 'age']:
    if col in df.columns:
        numeric_field = col
        break
else:
    if possible_numeric_cols:
        numeric_field = possible_numeric_cols[0]
    else:
        numeric_field = df.columns[0]  # fallback

print(f"Using numeric field: {numeric_field}")

threshold = df[numeric_field].median()  # Use median as demonstration threshold

filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a demographic field, e.g., 'gender' or 'level_of_education' if present
group_fields = ['gender', 'level_of_education', 'village', 'marital_status']
group_field = None
for field in group_fields:
    if field in df.columns:
        group_field = field
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by '{group_field}', mean {numeric_field} per group:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group if group_field exists
if group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Using `mlcroissant`, we've loaded and explored the FAIR² Mental Health Survey dataset from Kilifi County, Kenya. We've demonstrated how to reference and extract records using Croissant schema `@id`s, performed basic EDA, filtered and normalized numeric mental health assessment scores, and visualized distributions and groupings. This approach ensures reproducibility, schema-aware data wrangling, and makes the workflow AI/ML-ready.

**You can extend this analysis by referencing additional field `@id`s, adding more EDA visualizations, or exporting processed tables for downstream ML tasks.**